In [2]:
from groq import Groq
import os

# Using the raw API key directly to initialize the client
api_key = "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"
client = Groq(api_key=api_key)
print("Groq connected successfully!")

Groq connected successfully!


In [3]:
%%writefile nutrition_mcp_server.py
import json
import sys

NUTRITION_DB = {
    "chicken breast": {"calories_per_100g": 165, "protein_per_100g": 31.0},
    "greek yogurt": {"calories_per_100g": 59, "protein_per_100g": 10.0},
    "tofu": {"calories_per_100g": 76, "protein_per_100g": 8.0},
    "lentils": {"calories_per_100g": 116, "protein_per_100g": 9.0},
    "rolled oats": {"calories_per_100g": 389, "protein_per_100g": 16.9},
    "almonds": {"calories_per_100g": 579, "protein_per_100g": 21.2},
    "paneer": {"calories_per_100g": 265, "protein_per_100g": 18.3},
}

TOOLS = [
    {
        "name": "get_calories",
        "description": "Returns the calorie information (kcal) for a given food item and optional weight in grams (default 100g).",
        "inputSchema": {
            "type": "object",
            "properties": {
                "food": {"type": "string", "description": "Name of the food item, e.g. 'chicken breast'"},
                "grams": {"type": "number", "description": "Weight in grams (default: 100)", "default": 100}
            },
            "required": ["food"]
        }
    },
    {
        "name": "get_protein",
        "description": "Returns the protein information (grams) for a given food item and optional weight in grams (default 100g).",
        "inputSchema": {
            "type": "object",
            "properties": {
                "food": {"type": "string", "description": "Name of the food item, e.g. 'chicken breast'"},
                "grams": {"type": "number", "description": "Weight in grams (default: 100)", "default": 100}
            },
            "required": ["food"]
        }
    }
]

def normalize_food(name: str):
    key = name.strip().lower()
    if key in NUTRITION_DB:
        return key, NUTRITION_DB[key]
    for k, v in NUTRITION_DB.items():
        if k in key or key in k:
            return k, v
    return None, None

def run_tool(name: str, args: dict) -> str:
    food_raw = args.get("food", "")
    grams = float(args.get("grams", 100))
    factor = grams / 100.0
    matched_key, entry = normalize_food(food_raw)

    if not entry:
        return f"No nutrition data found for '{food_raw}'. Available items: {', '.join(NUTRITION_DB.keys())}."

    if name == "get_calories":
        kcal = round(entry["calories_per_100g"] * factor, 1)
        if kcal.is_integer():
            kcal = int(kcal)
        return f"{kcal} kcal"
    elif name == "get_protein":
        protein = round(entry["protein_per_100g"] * factor, 1)
        return f"{protein} g protein"
    else:
        return f"Unknown tool: {name}"

def handle_request(msg: dict):
    method = msg.get("method")
    msg_id = msg.get("id")

    if method == "initialize":
        return {
            "jsonrpc": "2.0",
            "id": msg_id,
            "result": {
                "protocolVersion": "2025-03-26",
                "capabilities": {"tools": {}},
                "serverInfo": {"name": "nutrition-mcp-server", "version": "1.0.0"}
            }
        }
    if method == "notifications/initialized":
        return None
    if method == "tools/list":
        return {"jsonrpc": "2.0", "id": msg_id, "result": {"tools": TOOLS}}
    if method == "tools/call":
        params = msg.get("params", {})
        tool_name = params.get("name")
        arguments = params.get("arguments", {})
        result_text = run_tool(tool_name, arguments)
        return {
            "jsonrpc": "2.0",
            "id": msg_id,
            "result": {
                "content": [{"type": "text", "text": result_text}]
            }
        }
    if msg_id is not None:
        return {
            "jsonrpc": "2.0",
            "id": msg_id,
            "error": {"code": -32601, "message": f"Method not found: {method}"}
        }
    return None

def main():
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            req = json.loads(line)
            resp = handle_request(req)
            if resp is not None:
                sys.stdout.write(json.dumps(resp) + "\n")
                sys.stdout.flush()
        except Exception as e:
            err = {
                "jsonrpc": "2.0",
                "id": None,
                "error": {"code": -32603, "message": str(e)}
            }
            sys.stdout.write(json.dumps(err) + "\n")
            sys.stdout.flush()

if __name__ == "__main__":
    main()

Writing nutrition_mcp_server.py


In [6]:
import json
import subprocess

class NutritionMCPClient:
    def __init__(self, server_script: str = "nutrition_mcp_server.py"):
        self.server_script = server_script
        self.proc = None
        self._req_id = 0
        self.tools = []

    def _next_id(self) -> int:
        self._req_id += 1
        return self._req_id

    def _send(self, payload: dict):
        self.proc.stdin.write(json.dumps(payload) + "\n")
        self.proc.stdin.flush()

    def _receive(self) -> dict:
        line = self.proc.stdout.readline()
        if not line:
            stderr_out = self.proc.stderr.read() if self.proc.stderr else ""
            raise RuntimeError(f"MCP server closed unexpectedly. stderr: {stderr_out}")
        return json.loads(line)

    def connect(self):
        self.proc = subprocess.Popen(
            ["python3", self.server_script],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1
        )
        init_resp = self._send_and_wait("initialize", {
            "protocolVersion": "2025-03-26",
            "capabilities": {},
            "clientInfo": {"name": "nutrition-ai-agent-client", "version": "1.0.0"}
        })
        self._send({"jsonrpc": "2.0", "method": "notifications/initialized"})
        server_info = init_resp.get("result", {}).get("serverInfo", {})
        print(f"Connected to MCP Server: {server_info.get('name')} v{server_info.get('version')}")
        return init_resp

    def _send_and_wait(self, method: str, params: dict = None) -> dict:
        req = {"jsonrpc": "2.0", "id": self._next_id(), "method": method}
        if params is not None:
            req["params"] = params
        self._send(req)
        return self._receive()

    def discover_tools(self):
        resp = self._send_and_wait("tools/list")
        self.tools = resp.get("result", {}).get("tools", [])
        print("Discovered MCP Tools:", [t["name"] for t in self.tools])
        return self.tools

    def to_groq_tools(self):
        return [
            {
                "type": "function",
                "function": {
                    "name": t["name"],
                    "description": t.get("description", ""),
                    "parameters": t.get("inputSchema", {"type": "object", "properties": {}})
                }
            }
            for t in self.tools
        ]

    def call_tool(self, name: str, arguments: dict) -> str:
        resp = self._send_and_wait("tools/call", {"name": name, "arguments": arguments})
        content = resp.get("result", {}).get("content", [])
        return "\n".join(item.get("text", "") for item in content if item.get("type") == "text")

    def close(self):
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            self.proc.wait(timeout=3)


conversation_history = [
    {
        "role": "system",
        "content": (
            "You are a helpful, context-aware nutritionist AI agent. "
            "Remember user details (name, goals, dietary preferences) across turns. "
            "Always call the available MCP tools when asked about calories or protein."
        )
    }
]


def ask_nutrition_agent(user_prompt: str, mcp_client: NutritionMCPClient, model: str = "openai/gpt-oss-20b") -> str:
    conversation_history.append({"role": "user", "content": user_prompt})
    groq_tools = mcp_client.to_groq_tools()

    while True:
        response = client.chat.completions.create(
            model=model,
            messages=conversation_history,
            tools=groq_tools,
            tool_choice="auto",
            temperature=0.3
        )
        message = response.choices[0].message
        tool_calls = getattr(message, "tool_calls", None) or []

        if not tool_calls:
            final_reply = message.content or ""
            conversation_history.append({"role": "assistant", "content": final_reply})
            return final_reply

        conversation_history.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                }
                for tc in tool_calls
            ]
        })

        for tc in tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments or "{}")
            print(f"[Agent] Calling MCP tool `{fn_name}` with {fn_args}")
            server_result = mcp_client.call_tool(fn_name, fn_args)
            print(f"[MCP Server] Returned: {server_result}")

            conversation_history.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": server_result
            })


mcp = NutritionMCPClient("nutrition_mcp_server.py")
mcp.connect()
mcp.discover_tools()

print("\n--- Test 1: Calories ---")
print("AI Agent:", ask_nutrition_agent("How many calories are in 100g of chicken breast?", mcp))

print("\n--- Test 2: Protein ---")
print("AI Agent:", ask_nutrition_agent("How much protein is in 100g of paneer?", mcp))



Connected to MCP Server: nutrition-mcp-server v1.0.0
Discovered MCP Tools: ['get_calories', 'get_protein']

--- Test 1: Calories ---
[Agent] Calling MCP tool `get_calories` with {'food': 'chicken breast', 'grams': 100}
[MCP Server] Returned: 165 kcal
AI Agent: A 100‑gram serving of skin‑less, boneless chicken breast contains about **165 kcal**.

--- Test 2: Protein ---
[Agent] Calling MCP tool `get_protein` with {'food': 'paneer', 'grams': 100}
[MCP Server] Returned: 18.3 g protein
AI Agent: A 100‑gram serving of paneer contains roughly **18.3 g of protein**.


In [7]:
print("\n--- Test 3: Context + Tool Call ---")
print("AI Agent:", ask_nutrition_agent("I'm building muscle on a vegetarian diet—how much protein is in 100g of paneer?", mcp))
mcp.close()


--- Test 3: Context + Tool Call ---
AI Agent: A 100‑gram serving of paneer contains about **18.3 g of protein**—a solid source of protein for a vegetarian muscle‑building diet.
